In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
meta_schema = StructType([
    StructField("main_category", StringType(), True),
    StructField("title", StringType(), True),
    StructField("average_rating", DoubleType(), True),
    StructField("rating_number", LongType(), True),
    StructField("features", ArrayType(StringType()), True),
    StructField("description", ArrayType(StringType()), True),
    StructField("price", DoubleType(), True),
    StructField("images", ArrayType(MapType(StringType(), StringType())), True),
    StructField("videos", ArrayType(MapType(StringType(), StringType())), True),
    StructField("store", StringType(), True),
    StructField("categories", ArrayType(StringType()), True),

    StructField("details", StringType(), True),

    StructField("parent_asin", StringType(), True),
    StructField("bought_together", DoubleType(), True),
    StructField("subtitle", StringType(), True),
    StructField("author", StringType(), True),
])
raw_meta_df = spark.read.text("/content/drive/MyDrive/data - MoMD/meta_Office_Products.jsonl")
meta_df = (
    raw_meta_df
    .select(from_json(col("value"), meta_schema).alias("data"))
    .select("data.*")
)

In [ ]:
df_data = spark.read.format("json").option("inferSchema", "true").option("header", "true").load("/content/drive/MyDrive/data - MoMD/Office_Products.jsonl")

In [ ]:
meta_df.show()

+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|       main_category|               title|average_rating|rating_number|             features|         description| price|              images|              videos|               store|          categories|             details|parent_asin|bought_together|            subtitle|author|
+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|     Office Products|Alliance Rubber 0...|           4.5|          665| [REUSABLE: These ...|[Alliance Rubber ...|  2.68|[{thumb -> https:...|[{tit

In [ ]:
df_data.show()

+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B01AHHL4X2|           0|                  []| B01MZ3SD2X|   5.0|Lovely ink. Write...|1677939345945| Pretty & I love it!|AFKZENTNBQ7A7V7UX...|             true|
|B08L6H23JZ|           0|                  []| B08L6H23JZ|   4.0|Overall I’m prett...|1677939160682|2 excellent 1 ext...|AFKZENTNBQ7A7V7UX...|             true|
|B07JDZ5J46|           2|                  []| B07JDZ5J46|   1.0|[[VIDEOID:63276c1...|1660188831933|I don’t get the r...|AFKZENTNBQ7A7V7UX...|             true|
|B004MNX7EW|           0|[{IMAGE, 

Lọc cột cần thiết

In [ ]:
from pyspark.sql.functions import col

In [ ]:
meta_df_raw = meta_df
df_data_raw = df_data

In [ ]:
meta_df = meta_df.select("main_category","title","features","description","price","parent_asin")
df_data = df_data.select("asin","parent_asin","rating","timestamp","user_id","verified_purchase")

In [ ]:
meta_df.printSchema()

root
 |-- main_category: string (nullable = true)
 |-- title: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- bought_together: double (nullable = true)



In [ ]:
meta_df.count()

710503

In [ ]:
df_data.printSchema()

root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [ ]:
df_data.count()

12845712

In [ ]:
df_data = df_data.withColumnsRenamed({
    'asin':'product_id',
    'parent_asin':'product_parent',
    'rating':'review_rating',
    'timestamp':'review_date'
})

In [ ]:
user_counts = df_data.groupBy("user_id").agg(count("*").alias("number_rating"))
active_users = user_counts.filter(col("number_rating") >= 5)
active_users = active_users.drop("number_rating")
df_data = df_data.join(active_users, "user_id")

In [ ]:
product_counts = df_data.groupBy("product_id").agg(count("*").alias("number_rating"))
popular_products = product_counts.filter(col("number_rating") >= 10)
popular_products = popular_products.drop("number_rating")
df_data = df_data.join(popular_products, "product_id")

In [ ]:
df_data.count()

1827847

In [ ]:
output_path = "/content/drive/MyDrive/data - MoMD"
df_data.write.mode("overwrite").parquet(output_path)

In [ ]:
output_path = "/content/drive/MyDrive/data_parquet/rating_data"
df_data_clean = spark.read.parquet(output_path)

In [ ]:
df_data_clean.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_date: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [ ]:
df_data_clean.show()

+----------+--------------------+--------------+-------------+-------------+-----------------+
|product_id|             user_id|product_parent|review_rating|  review_date|verified_purchase|
+----------+--------------------+--------------+-------------+-------------+-----------------+
|0060883286|AFQKOTM5J6GZJD55J...|    0060883286|          5.0|1595824788124|             true|
|0060883286|AFK6NT5GDAGC75OVP...|    0060883286|          5.0|1410294787000|             true|
|0060883286|AGR5UX66HZNA5KON7...|    0060883286|          5.0|1547680707999|             true|
|0060883286|AFESUS3IVBSSEBJQT...|    0060883286|          5.0|1413418070000|            false|
|0060883286|AFTLUVGQWKW6XSQ5T...|    0060883286|          5.0|1568486133656|             true|
|0060883286|AG5B535BFCJIJEA47...|    0060883286|          1.0|1222036210000|            false|
|0060883286|AHO2E4OI7E3BOGAER...|    0060883286|          5.0|1403710554000|             true|
|0060883286|AHWQI5RDKU3AHHGKA...|    0060883286|  

In [ ]:
df_data_clean.select("product_id").distinct().count()

54659

In [ ]:
from pyspark.sql.functions import col, from_unixtime

data = [(1677939345945,), (1677939160682,)]
df = spark.createDataFrame(data,["timestamp_ms"])

df = df.withColumn(
    "timestamp",
    from_unixtime(col("timestamp_ms") / 1000)
)

df.show(truncate=False)

+-------------+-------------------+
|timestamp_ms |timestamp          |
+-------------+-------------------+
|1677939345945|2023-03-04 14:15:45|
|1677939160682|2023-03-04 14:12:40|
+-------------+-------------------+



In [ ]:
from pyspark.ml.recommendation import ALS

In [ ]:
output_path = "/content/drive/MyDrive/data_parquet/rating_data"
df_data_clean = spark.read.parquet(output_path)

In [ ]:
df_data_clean.show()

+----------+--------------------+--------------+-------------+-------------+-----------------+
|product_id|             user_id|product_parent|review_rating|  review_date|verified_purchase|
+----------+--------------------+--------------+-------------+-------------+-----------------+
|0060883286|AFQKOTM5J6GZJD55J...|    0060883286|          5.0|1595824788124|             true|
|0060883286|AFK6NT5GDAGC75OVP...|    0060883286|          5.0|1410294787000|             true|
|0060883286|AGR5UX66HZNA5KON7...|    0060883286|          5.0|1547680707999|             true|
|0060883286|AFESUS3IVBSSEBJQT...|    0060883286|          5.0|1413418070000|            false|
|0060883286|AFTLUVGQWKW6XSQ5T...|    0060883286|          5.0|1568486133656|             true|
|0060883286|AG5B535BFCJIJEA47...|    0060883286|          1.0|1222036210000|            false|
|0060883286|AHO2E4OI7E3BOGAER...|    0060883286|          5.0|1403710554000|             true|
|0060883286|AHWQI5RDKU3AHHGKA...|    0060883286|  

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import explode, expr, collect_list
from pyspark.mllib.evaluation import RankingMetrics

In [ ]:
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_index")
product_indexer = StringIndexer(inputCol="product_id", outputCol="product_index")

user_indexer_model = user_indexer.fit(df_data_clean)
product_indexer_model = product_indexer.fit(df_data_clean)
df_data_clean = user_indexer_model.transform(df_data_clean)
df_data_clean = product_indexer_model.transform(df_data_clean)

In [ ]:
df_data_clean = df_data_clean.withColumn("user_index", col("user_index").cast("int"))
df_data_clean = df_data_clean.withColumn("product_index", col("product_index").cast("int"))

In [ ]:
df_data_clean.show()

+----------+--------------------+--------------+-------------+-------------+-----------------+----------+-------------+
|product_id|             user_id|product_parent|review_rating|  review_date|verified_purchase|user_index|product_index|
+----------+--------------------+--------------+-------------+-------------+-----------------+----------+-------------+
|0060883286|AFQKOTM5J6GZJD55J...|    0060883286|          5.0|1595824788124|             true|    135911|        13048|
|0060883286|AFK6NT5GDAGC75OVP...|    0060883286|          5.0|1410294787000|             true|    194692|        13048|
|0060883286|AGR5UX66HZNA5KON7...|    0060883286|          5.0|1547680707999|             true|     99525|        13048|
|0060883286|AFESUS3IVBSSEBJQT...|    0060883286|          5.0|1413418070000|            false|    191934|        13048|
|0060883286|AFTLUVGQWKW6XSQ5T...|    0060883286|          5.0|1568486133656|             true|     17635|        13048|
|0060883286|AG5B535BFCJIJEA47...|    006

In [ ]:
df_data_clean_train = df_data_clean.select("user_index", "product_index", "review_rating", "review_date").sort("review_date")

In [ ]:
quantiles = df_data_clean_train.approxQuantile("review_date", [0.8], 0)
split_time = quantiles[0]

train_df = df_data_clean_train.filter(col("review_date") <= split_time)
test_df = df_data_clean_train.filter(col("review_date") > split_time)

In [ ]:
k = 20

In [ ]:
als = ALS(userCol = "user_index",
          itemCol = "product_index",
          ratingCol = "review_rating",
          coldStartStrategy = "drop",
          rank = 50,
          maxIter=10,
          regParam=0.1)

In [ ]:
model = als.fit(train_df)

In [ ]:
df_rec_user = model.recommendForAllUsers(k)

df_rec_user_exp = df_rec_user.select(
    "user_index",
    explode("recommendations").alias("recommendation")
)

df_rec_user_exp = df_rec_user_exp.select(
    "user_index",
    "recommendation.product_index",
    "recommendation.rating"
)

In [ ]:
df_rec_user2 = df_rec_user
df_rec_user2.show()

+----------+--------------------+
|user_index|     recommendations|
+----------+--------------------+
|        26|[{16019, 5.667464...|
|        27|[{27339, 6.102278...|
|        28|[{23688, 6.388294...|
|        31|[{37360, 6.252848...|
|        34|[{53607, 5.870527...|
|        53|[{43808, 5.693413...|
|        65|[{23688, 7.391992...|
|        76|[{23688, 6.992546...|
|        78|[{40033, 6.287309...|
|        81|[{53204, 5.947049...|
|        85|[{48078, 6.376401...|
|       101|[{27339, 5.904172...|
|       108|[{53046, 7.199929...|
|       115|[{23688, 6.978523...|
|       126|[{37072, 3.397951...|
|       133|[{23688, 6.533725...|
|       137|[{43808, 6.825073...|
|       148|[{27339, 5.559210...|
|       183|[{44315, 6.966765...|
|       193|[{27339, 6.018884...|
+----------+--------------------+
only showing top 20 rows


In [ ]:
df_rec_user_exp.show(truncate = False)

+----------+-------------+---------+
|user_index|product_index|rating   |
+----------+-------------+---------+
|26        |16019        |5.667464 |
|26        |52711        |5.6291533|
|26        |53088        |5.61875  |
|26        |53204        |5.575192 |
|26        |34406        |5.547359 |
|26        |31868        |5.5350814|
|26        |23688        |5.5341706|
|26        |33880        |5.4906664|
|26        |47801        |5.483678 |
|26        |27339        |5.471818 |
|26        |40440        |5.4645486|
|26        |48464        |5.4616294|
|26        |25423        |5.4371114|
|26        |46254        |5.4188604|
|26        |52726        |5.3892493|
|26        |14013        |5.3795123|
|26        |33222        |5.375397 |
|26        |50733        |5.365218 |
|26        |16038        |5.3630085|
|26        |13519        |5.3594103|
+----------+-------------+---------+
only showing top 20 rows


In [ ]:
df_rec_user_pred = df_rec_user.select(
    col("user_index"),
    expr("transform(recommendations, x -> x.product_index)").alias("pred_items")
)

In [ ]:
true_df = test_df.groupBy("user_index").agg(
    collect_list("product_index").alias("true_items")
)

In [ ]:
eval_df = df_rec_user_pred.join(true_df, "user_index")

In [ ]:
predictionAndLabels = eval_df.select(
    "pred_items",
    "true_items"
).rdd.map(lambda x: (x[0], x[1]))

> filter item đã xuất hiện trong train

In [ ]:
metrics = RankingMetrics(predictionAndLabels)

precision = metrics.precisionAt(k)
recall = metrics.recallAt(k)
map_score = metrics.meanAveragePrecision

print("Precision@10:", precision)
print("Recall@10:", recall)
print("MAP:", map_score)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Precision@10: 4.10888072242431e-05
Recall@10: 0.00045021693115680214
MAP: 8.412700254400196e-05
